# 05 — Validation, Stress Tests & Final Analysis (UBCF)

Phase 5:
- Bootstrap CIs (100 iterations)
- Shot noise stress (1024 shots)
- IQP-style entanglement stress variant
- Lloyd complexity/privacy analysis
- Paper-ready plots and addendum export

Output: `research_addendum_ubcf.csv`

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import qiskit
    QISKIT_AVAILABLE = True
except Exception:
    QISKIT_AVAILABLE = False

IN_DIR = Path("data/processed_32m_ubcf")
OUT_DIR = Path("data/processed_32m_ubcf")

SEED = 42
BOOTSTRAP_ITERS = 100
SHOTS = 1024

ubcf_metrics = pd.read_csv(IN_DIR / "ubcf_metrics.csv") if (IN_DIR / "ubcf_metrics.csv").exists() else pd.DataFrame()
stress_df = pd.read_csv(IN_DIR / "hybrid_ubcf_stress.csv") if (IN_DIR / "hybrid_ubcf_stress.csv").exists() else pd.DataFrame()
kernel_data = np.load(IN_DIR / "user_kernel_matrix.npz")
kernel = kernel_data["kernel"].astype(np.float32)

print(QISKIT_AVAILABLE, kernel.shape)

In [ ]:
# Bootstrap CI on sparse regime metrics
rng = np.random.default_rng(SEED)
sparse = stress_df[stress_df["sparsity_drop"] >= 0.80].copy() if not stress_df.empty else pd.DataFrame()

def bootstrap_ci(vals, n_iter=100):
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    boots = []
    for _ in range(n_iter):
        s = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(s))
    boots = np.array(boots, dtype=np.float64)
    return float(np.mean(vals)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

hr_mean, hr_low, hr_high = bootstrap_ci(sparse["hr@10"].dropna().values if not sparse.empty else np.array([]), BOOTSTRAP_ITERS)
nd_mean, nd_low, nd_high = bootstrap_ci(sparse["ndcg@10"].dropna().values if not sparse.empty else np.array([]), BOOTSTRAP_ITERS)

ci_df = pd.DataFrame([
    {"metric": "hr@10", "mean": hr_mean, "ci_low": hr_low, "ci_high": hr_high, "iters": BOOTSTRAP_ITERS},
    {"metric": "ndcg@10", "mean": nd_mean, "ci_low": nd_low, "ci_high": nd_high, "iters": BOOTSTRAP_ITERS},
])
print(ci_df)

In [ ]:
# Shot noise (1024) and IQP stress variants on kernel statistics
rng_q = np.random.default_rng(SEED)
k0 = np.clip(kernel, 0.0, 1.0)
k_shot = rng_q.binomial(SHOTS, k0) / SHOTS
k_iqp = np.cos(np.pi * (1.0 - k0)) ** 2 if QISKIT_AVAILABLE else np.power(k0, 1.5)
k_iqp = np.clip(k_iqp, 0.0, 1.0)

stress_kernel_df = pd.DataFrame([
    {"variant": "original", "mean_similarity": float(np.mean(k0)), "std_similarity": float(np.std(k0))},
    {"variant": f"shot_noise_{SHOTS}", "mean_similarity": float(np.mean(k_shot)), "std_similarity": float(np.std(k_shot))},
    {"variant": "iqp_variant", "mean_similarity": float(np.mean(k_iqp)), "std_similarity": float(np.std(k_iqp))},
])
print(stress_kernel_df)

In [ ]:
# Complexity and privacy logging (Lloyd-style query accounting)
def lloyd_query_log(kernel_mat, k=6, iters=8, seed=42):
    rng_local = np.random.default_rng(seed)
    n = kernel_mat.shape[0]
    centers = rng_local.choice(n, size=min(k, n), replace=False)
    labels = np.zeros(n, dtype=np.int32)
    logs = []
    for it in range(iters):
        t0 = time.perf_counter()
        queries = int(n * len(centers) + len(centers))
        d2 = np.zeros((n, len(centers)), dtype=np.float32)
        dg = np.diag(kernel_mat)
        for c, ci in enumerate(centers):
            d2[:, c] = dg + kernel_mat[ci, ci] - 2.0 * kernel_mat[:, ci]
        labels = np.argmin(d2, axis=1).astype(np.int32)
        logs.append({"iter": int(it+1), "n": int(n), "k": int(len(centers)), "queries": queries, "elapsed_sec": float(time.perf_counter()-t0)})
    return pd.DataFrame(logs)

priv_df = lloyd_query_log(k0, k=6, iters=10, seed=SEED)

runtime_rows = []
for n_sub in [50, 100, 150, 200, 250, min(300, k0.shape[0])]:
    sub = k0[:n_sub, :n_sub]
    t0 = time.perf_counter()
    _ = lloyd_query_log(sub, k=min(6, max(2, n_sub//30)), iters=5, seed=SEED)
    runtime_rows.append({"n": int(n_sub), "runtime_sec": float(time.perf_counter()-t0), "log_term": float(np.log(max(2, n_sub*n_sub)))})
runtime_df = pd.DataFrame(runtime_rows)

add_rows = [
    {"section": "bootstrap", "metric": "hr@10", "value": hr_mean, "ci_low": hr_low, "ci_high": hr_high},
    {"section": "bootstrap", "metric": "ndcg@10", "value": nd_mean, "ci_low": nd_low, "ci_high": nd_high},
    {"section": "privacy", "metric": "total_queries", "value": float(priv_df["queries"].sum()), "ci_low": np.nan, "ci_high": np.nan},
]
for _, r in stress_kernel_df.iterrows():
    add_rows.append({"section": "quantum_stress", "metric": f"{r['variant']}_mean_similarity", "value": float(r['mean_similarity']), "ci_low": np.nan, "ci_high": np.nan})

research_addendum_ubcf = pd.DataFrame(add_rows)
research_addendum_ubcf.to_csv(OUT_DIR / "research_addendum_ubcf.csv", index=False)
priv_df.to_csv(OUT_DIR / "privacy_log_ubcf.csv", index=False)
runtime_df.to_csv(OUT_DIR / "runtime_complexity_ubcf.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if not stress_df.empty:
    axes[0].plot(stress_df["sparsity_drop"]*100, stress_df["hr@10"], marker="o", label="HR@10")
    axes[0].plot(stress_df["sparsity_drop"]*100, stress_df["ndcg@10"], marker="o", label="NDCG@10")
axes[0].set_title("Sparse Retention (UBCF)")
axes[0].set_xlabel("Drop %")
axes[0].set_ylabel("Metric")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(runtime_df["n"], runtime_df["runtime_sec"], marker="o", label="Observed")
axes[1].plot(runtime_df["n"], runtime_df["log_term"] / runtime_df["log_term"].max() * runtime_df["runtime_sec"].max(), linestyle="--", label="Scaled log(MN)")
axes[1].set_title("Runtime / Complexity Trend")
axes[1].set_xlabel("n users (subset)")
axes[1].set_ylabel("seconds")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print("Saved: research_addendum_ubcf.csv")